In [ ]:
# SHARED — support-ticket domain + FakeLLM + TicketQuery + canonical TicketAgent.
# Identical to module-11; this is the system under test. NO API key, NO network.
from __future__ import annotations
import json
from types import SimpleNamespace
from typing import Optional, Any, Callable
from dataclasses import dataclass, field
from pydantic import BaseModel, Field, ValidationError

@dataclass
class SupportTicket:
    id: int
    subject: str
    category: str          # "Billing" | "Technical" | "Account"
    priority: str          # "low" | "normal" | "high" | "urgent"
    is_open: bool = True
    tags: list[str] = field(default_factory=list)

SAMPLE_TICKETS = [
    SupportTicket(1, "Can't log in after password reset", "Account", "high"),
    SupportTicket(2, "Invoice double-charged this month",  "Billing", "urgent"),
    SupportTicket(3, "How do I export my data?",            "Technical", "low"),
    SupportTicket(4, "App crashes on upload",               "Technical", "high"),
    SupportTicket(5, "Refund not received",                 "Billing", "normal", is_open=False),
]

# --- the dependency-injection seam: a fake stand-in for openai.OpenAI ---
def _msg(content=None, tool_calls=None):
    return SimpleNamespace(content=content, tool_calls=tool_calls)
def _resp(message):
    return SimpleNamespace(choices=[SimpleNamespace(message=message)])
def _tool_call(call_id, name, **args):
    return SimpleNamespace(id=call_id,
        function=SimpleNamespace(name=name, arguments=json.dumps(args)))

class FakeLLM:
    """Scripts chat.completions.create — same shape as openai.OpenAI, no API key.
    Exactly the Day-3 pattern: inject a fake at the seam instead of the real client."""
    def __init__(self, responses):
        self._responses = list(responses)
        self.chat = SimpleNamespace(
            completions=SimpleNamespace(create=self._create))
    def _create(self, **kwargs):
        return self._responses.pop(0)

class TicketQuery(BaseModel):
    category: Optional[str] = Field(default=None, description="Restrict to this category, or null for all.")
    priority: Optional[str] = Field(default=None, description="One of low|normal|high|urgent, or null.")
    open_only: bool = False
    subject_contains: Optional[str] = Field(default=None, description="Substring the subject must contain.")

class AgentError(RuntimeError):
    pass

@dataclass
class ToolSpec:
    name: str
    description: str
    parameters: dict
    fn: Callable[..., Any]
    def to_openai_schema(self) -> dict:
        return {"type": "function", "function": {
            "name": self.name, "description": self.description,
            "parameters": self.parameters}}

class ToolRegistry:
    def __init__(self):
        self._tools: dict[str, ToolSpec] = {}
    def tool(self, *, name, description, parameters):
        def deco(fn):
            self._tools[name] = ToolSpec(name, description, parameters, fn)
            return fn
        return deco
    def get(self, name) -> ToolSpec:
        if name not in self._tools:
            raise KeyError(f"tool {name!r} not registered")
        return self._tools[name]
    def openai_schemas(self):
        return [t.to_openai_schema() for t in self._tools.values()]

@dataclass
class ToolCallRecord:
    tool: str
    arguments: dict
    result: Any

@dataclass
class AgentResult:
    answer: str
    tool_calls: list
    steps: int

SYSTEM_PROMPT = ("You are a support-desk assistant. Use the tools to answer "
                 "questions about support tickets. Prefer one accurate tool call.")

def _assistant_turn(msg) -> dict:
    """Re-encode the model's tool-call turn so the next API call accepts it."""
    return {"role": "assistant", "content": msg.content,
            "tool_calls": [{"id": tc.id, "type": "function",
                "function": {"name": tc.function.name,
                             "arguments": tc.function.arguments}}
                for tc in msg.tool_calls]}

def _tool_reply(call, result) -> dict:
    """One tool message per call, tagged with its tool_call_id — the chaining
    slide 11.5 warns about: every reply points back at the call that asked."""
    return {"role": "tool", "tool_call_id": call.id,
            "name": call.function.name,
            "content": json.dumps(result, default=str)}

class TicketAgent:
    def __init__(self, store, llm_client, *, model="gpt-4o-mini", max_steps=5):
        self.store = store
        self.llm = llm_client
        self.model = model
        self.max_steps = max_steps
        self.registry = self._build_registry()

    def _build_registry(self) -> ToolRegistry:
        reg = ToolRegistry()

        @reg.tool(name="list_open_tickets",
                  description="List all currently open tickets.",
                  parameters={"type": "object", "properties": {}, "required": []})
        def list_open_tickets():
            return [t.__dict__ for t in self.store if t.is_open]

        @reg.tool(name="search_tickets",
                  description="Find tickets whose subject contains a substring (case-insensitive).",
                  parameters={"type": "object",
                              "properties": {"term": {"type": "string"}},
                              "required": ["term"]})
        def search_tickets(term):
            return [t.__dict__ for t in self.store if term.lower() in t.subject.lower()]

        @reg.tool(name="count_by_priority",
                  description="Count tickets grouped by priority.",
                  parameters={"type": "object", "properties": {}, "required": []})
        def count_by_priority():
            out: dict[str, int] = {}
            for t in self.store:
                out[t.priority] = out.get(t.priority, 0) + 1
            return out

        return reg

    def ask(self, user_prompt: str) -> AgentResult:
        messages = [{"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}]
        log = []
        for step in range(1, self.max_steps + 1):
            # PLAN -- model sees the conversation + tool schemas, then replies
            resp = self.llm.chat.completions.create(
                model=self.model, messages=messages,
                tools=self.registry.openai_schemas())
            msg = resp.choices[0].message

            if not (msg.tool_calls or []):              # no tools asked -> final answer
                return AgentResult((msg.content or "").strip(), log, step)

            messages.append(_assistant_turn(msg))       # record the model's request
            # ACT + OBSERVE -- run each requested tool, feed its result back
            for call in msg.tool_calls:
                result = self._invoke(call.function.name, call.function.arguments)
                log.append(ToolCallRecord(call.function.name,
                    json.loads(call.function.arguments or "{}"), result))
                messages.append(_tool_reply(call, result))

        raise AgentError(f"did not converge in {self.max_steps} steps")

    def _invoke(self, name, args_json):
        try:
            spec = self.registry.get(name)
        except KeyError:                       # unknown tool -> error observation, not a crash
            return {"error": f"unknown tool: {name!r}"}
        return spec.fn(**(json.loads(args_json or "{}")))

# tiny in-notebook test harness (NOT pytest — just asserts you can watch run)
def check(label, cond):
    print(("PASS" if cond else "FAIL"), label)
    assert cond

# Module 12 ⭐ — Code Along: Testing & validating AI

An LLM is **non-deterministic**: the same prompt yields a *distribution* of outputs. You can't assert on the prose. So we never test the wording — we test **four other things**, each a different class of check:

| Class | What it pins down | Needs an API key? |
|---|---|---|
| 1 · Tool tests | the deterministic Python the agent calls | no |
| 2 · Schema tests | the LLM's JSON validates against Pydantic | no |
| 3 · Loop tests | the orchestration — calls, steps, guards (mock the LLM) | no |
| 4 · Golden evals | locked behaviour on named cases | no (mocked) |

Assert on **shape, behaviour, substring, bounds** — never on the exact sentence. Three of the four classes need no key and run in the **same Day-3 CI**. Each cell below builds an agent, exercises it, and asserts with `check(...)`.

In [ ]:
# CLASS 1 — tool tests: the tools are plain Python, so test them like any function.
# No LLM involved at all — FakeLLM([]) is never called because we reach past it
# into the registry and invoke the tool's .fn directly.
agent = TicketAgent(SAMPLE_TICKETS, FakeLLM([]))

search = agent.registry.get("search_tickets").fn
check("search is case-insensitive", search(term="RESET")[0]["id"] == 1)
check("search misses unrelated terms", search(term="quantum") == [])

counts = agent.registry.get("count_by_priority").fn()
check("count_by_priority", counts == {"high": 2, "urgent": 1, "low": 1, "normal": 1})

open_ids = [t["id"] for t in agent.registry.get("list_open_tickets").fn()]
check("closed ticket 5 is excluded", open_ids == [1, 2, 3, 4])

## Class 2 — schema tests

When the LLM returns JSON (the Module 10 extraction task), validate it against the **same Pydantic discipline from Day 2**. Good JSON parses into a typed object; malformed JSON raises a structured `ValidationError` you can catch — the model is your contract.

In [ ]:
# CLASS 2 — schema tests: validate the LLM's JSON against TicketQuery.
# Optional fields default cleanly when the model omits them.
check("optional fields default to None", TicketQuery().category is None)
check("open_only defaults False", TicketQuery().open_only is False)

# Valid JSON parses into a typed object.
q = TicketQuery.model_validate_json('{"category": "Billing", "open_only": true}')
check("valid JSON parses", q.category == "Billing" and q.open_only is True)

# Invalid payload — open_only must be a bool, not the string "maybe" — must raise.
raised = False
try:
    TicketQuery.model_validate({"open_only": "maybe"})
except ValidationError as e:
    raised = True
    print("caught ValidationError ->", e.errors()[0]["loc"], e.errors()[0]["type"])
check("bad type raises ValidationError", raised)

## Class 3 — loop tests

Mock the LLM with a **scripted `FakeLLM`** (the Day-3 déjà vu: inject a fake at the seam). Now the orchestration is deterministic, so we assert on *which* tools fired and *how many* steps ran — never on the prose.

In [ ]:
# CLASS 3 — loop test: one tool call, then a final answer. Two LLM turns = two steps.
agent = TicketAgent(SAMPLE_TICKETS, FakeLLM([
    _resp(_msg(tool_calls=[_tool_call("c1", "count_by_priority")])),
    _resp(_msg(content="There are 2 high, 1 urgent, 1 low and 1 normal ticket.")),
]))
r = agent.ask("how many tickets by priority?")

check("exactly one tool call", [c.tool for c in r.tool_calls] == ["count_by_priority"])
check("two steps (tool turn + answer turn)", r.steps == 2)
check("tool result fed back into the loop", r.tool_calls[0].result["high"] == 2)

## Class 3b — runaway protection & graceful errors

Two failure modes the loop must survive: an LLM that **never stops calling tools** (must hit `max_steps` and raise, not spin forever), and a call to a **tool that doesn't exist** (must return an error observation, not crash).

In [ ]:
# Runaway: an LLM that ALWAYS asks for another tool call. The loop must give up at max_steps.
class AlwaysToolLLM:
    """Never returns a final answer — every turn is another tool call."""
    def __init__(self):
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))
    def _create(self, **kwargs):
        return _resp(_msg(tool_calls=[_tool_call("loop", "count_by_priority")]))

agent = TicketAgent(SAMPLE_TICKETS, AlwaysToolLLM(), max_steps=3)
raised = False
try:
    agent.ask("spin forever")
except AgentError as e:
    raised = True
    print("caught ->", e)
check("max_steps raises AgentError", raised)

# Unknown tool: the model hallucinates a tool name. The loop records an error, then answers.
agent = TicketAgent(SAMPLE_TICKETS, FakeLLM([
    _resp(_msg(tool_calls=[_tool_call("c1", "does_not_exist")])),
    _resp(_msg(content="I couldn't find a tool for that.")),
]))
r = agent.ask("use a made-up tool")
check("unknown tool returns an error, no crash", "error" in r.tool_calls[0].result)

## Class 4 — golden evals

A small JSON file of **named cases** locks behaviour: each case names a prompt, the tool sequence it should drive, and substrings the answer must contain. We **parametrize** over the cases — one loop, many checks. In real CI this lives in `tests/evals/golden_queries.json` and runs under the `eval` marker.

In [ ]:
# CLASS 4 — golden evals. THIS is the cell the deck points to (module-12 cell 10):
# the mocked agent loop driven over a table of golden cases.
GOLDEN = [
    {"id": "t-01", "prompt": "how many by priority?",
     "expected_tool_calls": ["count_by_priority"], "tool_args": [{}],
     "expected_answer_contains": ["high"]},
    {"id": "t-02", "prompt": "find login issues",
     "expected_tool_calls": ["search_tickets"], "tool_args": [{"term": "log in"}],
     "expected_answer_contains": ["log in"]},
]

# A scripted answer that satisfies every needle for a case (so the eval is deterministic).
ANSWERS = {
    "t-01": "By priority: 2 high, 1 urgent, 1 low, 1 normal.",
    "t-02": "Ticket 1 is about not being able to log in after a password reset.",
}

for case in GOLDEN:
    # Build a FakeLLM: one tool call (with its args) per expected tool, then the answer.
    script = [_resp(_msg(tool_calls=[_tool_call(f"c{i}", name, **args)]))
              for i, (name, args) in enumerate(zip(case["expected_tool_calls"], case["tool_args"]))]
    script.append(_resp(_msg(content=ANSWERS[case["id"]])))
    agent = TicketAgent(SAMPLE_TICKETS, FakeLLM(script))

    r = agent.ask(case["prompt"])
    got_tools = [c.tool for c in r.tool_calls]
    check(f"[{case['id']}] tool sequence", got_tools == case["expected_tool_calls"])
    for needle in case["expected_answer_contains"]:
        check(f"[{case['id']}] answer contains {needle!r}", needle in r.answer)

## Recap

- **Test shape, behaviour, substring, bounds — never the prose.** The LLM is a distribution; the wording isn't a contract.
- **3 of the 4 classes need no API key** — tools, schema, and loop tests are pure Python + a mocked LLM. Only live golden evals would spend tokens; here even those are mocked.
- These run in the **same Day-3 CI** you already built — same `pytest`, same green bar.
- Every `FakeLLM` is the same dependency-injection seam as Day-3's `requests.Session`.

**→ Lab 12** writes these four classes as real `pytest` tests for the **`CatalogAgent`** (Product domain) — same patterns, your spine.